### PHASE 3 — Wednesday | Customer Risk Segmentation

---

*Note: For our collection workflows, we follow RBI guidelines which define default as loans 90+ days past due, regardless of status field.*

`Business Request from Collections Manager:` "My collection team needs a prioritized target list. Who are our riskiest customers? I need to know who has multiple defaults, who owes us the most money, and what action we should take."

`Wednesday Mindset Unlock:` A sorted list is not a strategy. Today you stop thinking like a developer producing ranked rows and start thinking like a collections manager who needs to tell their team of 20 people: call these 50 customers, in this order, using this specific approach for each segment. The output isn't a table — it's an operational tool. The difference is your custom columns.

`STRATEGIC ALERT — The National Verification Mandate:` The "National Portfolio" dataset (500k records) is now live in Databricks. (See the Setup Guide if you haven't yet). Preliminary checks suggest our regional samples (5k) may have been misleading.

`Your Mandate Today:` 

1. **Re-Verify Day 1 & 2**: Run your "Top 3 High Risk Cities" query on the 500k dataset. Does the risk epicentre stay the same? 
2. **Solve the Divergence**: One city that looked "Safe" in Day 2 now looks "Critical". Identify it and explain why the volume change revealed this. 
3. **Design, Don't Just Query**: The CEO wants a new **"Risk Efficiency"** metric. You must define the logic, not just the code.

---

### AI Thinking Assistant — Multi-Tool Suite

Use this multi-tool approach to solve the divergence:

- Antigravity (Strategy): "The CEO wants a 'Risk Efficiency' metric. How can I combine Default Rate with Total Financial Exposure to find the 'Most Expensive' cities, not just the 'Most Defaulters'?"

- Notebook LM: Upload Day 1-3 Memos. Ask: "What are the conflicting signals between regional and national reports?"

- Databricks AI: "Optimize this JOIN between 500k loans and 50k customers for partition performance."

---

Dataset Identification

- Who are our riskiest customers?
- who has multiple defaults?
- who owes us the most money?

> Identified datasets: *loans.csv*, *customers.csv*

---

### Start Spark Session

In [1]:
## Import dependencies and create Spark session
import time
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/01 09:08:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Global Variables

In [2]:
N = 20  # Number of rows to show in results
schema = "edufin_small"  # edufin_small or edufin_national

### Load the data

In [3]:
## Load the identified datasets and create a temporary views for SQL queries
loans = spark.read.csv(f"../datasets/{schema}/loans.csv", header=True)
customers = spark.read.csv(f"../datasets/{schema}/customers.csv", header=True)

loans.createOrReplaceTempView("loans")
customers.createOrReplaceTempView("customers")

print("Loans dataset:")
loans.show(n=5)
print("Customers dataset:")
customers.show(n=5)

Loans dataset:
+-------+-----------+--------------+-----------+-----------+-------------+------------------+----------------+-----------------+-------------+-----------+--------------------+
|loan_id|customer_id|institution_id|loan_amount|loan_status|interest_rate|loan_tenure_months|application_date|disbursement_date|maturity_date| emi_amount|     purpose_of_loan|
+-------+-----------+--------------+-----------+-----------+-------------+------------------+----------------+-----------------+-------------+-----------+--------------------+
|      1|       2440|          2954| 190607.125|  Defaulted|  11.39000034|                84|      08-12-2021|       23-12-2021|   16-11-2028|3302.540039|     Living Expenses|
|      2|       2440|          4741| 425798.375|     Active|  14.43999958|                48|      01-01-2022|       11-01-2022|   21-12-2025|11728.73047|Course Fees + Living|
|      3|       2440|           902|  318341.25|  Defaulted|  11.64999962|                96|      06-03-

Query 3A (BRD): Customer Profile Basics

---

- Build customer directory listing all customers with their loan activity summary.
- **Expected Output Columns:** Customer ID, customer name, loan count, total disbursed amounts
- **Business Question:** Who are our borrowers and how much have we lent to each?
- **Consideration:** Should you include customers with zero loans (LEFT JOIN) or only customers with loans (INNER JOIN)?

In [4]:
query = """
SELECT
    c.customer_id AS `Customer ID`,
    c.full_name AS `Customer Name`,
    COUNT(l.loan_id) AS `Loans`,
    CONCAT(ROUND(SUM(l.loan_amount) / 100000, 2), ' L') AS `Disbursed Amount`
FROM customers c
INNER JOIN loans l 
    ON c.customer_id = l.customer_id
GROUP BY c.customer_id, c.full_name
ORDER BY SUM(l.loan_amount) / 100000 DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------+----------------+-----+----------------+
|Customer ID|   Customer Name|Loans|Disbursed Amount|
+-----------+----------------+-----+----------------+
|       2479|      Upasna Pal|    3|         21.82 L|
|       4142|      Hemani Dey|    3|         21.62 L|
|       1755|     Harsh Baria|    3|         21.39 L|
|       3662|       Abha Kant|    3|         21.27 L|
|       3634|Ekansh Zachariah|    3|         21.17 L|
|       4737|     Arunima Din|    3|          21.1 L|
|       4075|  Jalsa Zacharia|    3|         21.06 L|
|        376|     Laban Ratta|    3|         20.93 L|
|       1353|    Diya Purohit|    3|         20.67 L|
|       2735|    Dipta Shukla|    3|         20.18 L|
|       3321|        Oni Date|    3|         20.09 L|
|       1245|      Ubika Bail|    3|         19.96 L|
|       1728|   Tristan Kumer|    3|         19.73 L|
|       2895|    Vansha Mitra|    3|         19.68 L|
|        611|  Ranveer Badami|    3|         19.64 L|
|       2198|       Nidhi Da

STEP 3A (Workbook) — The Scale Shift (Comparison Analysis)

---

What you're doing: Re-run your Day 2 "Default Rate by City" query on the new 500k dataset. Compare the results. This is the ultimate test of an analyst: recognizing when a sample size was too small to be trusted.

*Hint: SELECT city, COUNT(*) as volume, AVG(CASE WHEN status='Defaulted' THEN 1 ELSE 0 END) as rate... GROUP BY 1 ORDER BY 3 DESC*

In [5]:
customers_small = spark.read.csv("../datasets/edufin_small/customers.csv", header=True)
loans_small = spark.read.csv("../datasets/edufin_small/loans.csv", header=True)
dim_city_small = spark.read.csv("../datasets/edufin_small/dim_city.csv", header=True)

customers_national = spark.read.csv("../datasets/edufin_national/customers.csv", header=True)
loans_national = spark.read.csv("../datasets/edufin_national/loans.csv", header=True)
dim_city_national = spark.read.csv("../datasets/edufin_national/dim_city.csv", header=True)

customers_small.createOrReplaceTempView("customers_small")
loans_small.createOrReplaceTempView("loans_small")
dim_city_small.createOrReplaceTempView("dim_city_small")

customers_national.createOrReplaceTempView("customers_national")
loans_national.createOrReplaceTempView("loans_national")
dim_city_national.createOrReplaceTempView("dim_city_national")

In [27]:
query_small = """
SELECT 
    dc.city_name AS `City`,
    COUNT(*) AS `Total Loans`,
    ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END), 2) AS `Defaulted Loans`,
    CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*) * 100, 2), '%') AS `Default Rates (%)`
FROM loans_small l
INNER JOIN customers_small c ON l.customer_id = c.customer_id
INNER JOIN dim_city_small dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
HAVING COUNT(*) >= 100
ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*) DESC
LIMIT 10;
"""
query_national = """
SELECT 
    dc.city_name AS `City`,
    COUNT(*) AS `Total Loans`,
    ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END), 2) AS `Defaulted Loans`,
    CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*) * 100, 2), '%') AS `Default Rates (%)`
FROM loans_national l                         
INNER JOIN customers_national c ON l.customer_id = c.customer_id
INNER JOIN dim_city_national dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
HAVING COUNT(*) >= 100
ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*) DESC
LIMIT 10;
"""
spark.sql(query_small).show(truncate=False)
spark.sql(query_national).show(truncate=False)

+---------+-----------+---------------+-----------------+
|City     |Total Loans|Defaulted Loans|Default Rates (%)|
+---------+-----------+---------------+-----------------+
|Pune     |149        |30             |20.13%           |
|Ranchi   |168        |30             |17.86%           |
|Indore   |170        |28             |16.47%           |
|Jaipur   |160        |26             |16.25%           |
|Rajkot   |166        |26             |15.66%           |
|Kochi    |165        |25             |15.15%           |
|Ahmedabad|160        |24             |15.0%            |
|Jammu    |159        |23             |14.47%           |
|Nashik   |135        |19             |14.07%           |
|Lucknow  |179        |25             |13.97%           |
+---------+-----------+---------------+-----------------+

+-------------+-----------+---------------+-----------------+
|City         |Total Loans|Defaulted Loans|Default Rates (%)|
+-------------+-----------+---------------+-----------------+
|

Which city risk profile changed most? What was the % deafult rate on Day 2 vs Day 3?

*Pune had the most dramatic reversal, but Visakhapatnam is the most important discovery. In the small sample we had Pune at the top, but when compared to the result in the new 500k dataset, the top place is taken by Visakhapatnam, while Pune is nowhere to be found even in the top 10. Visakhapatnam changed the most in terms of discovery, it was invisible in the small sample (too few records to surface) but holds a 32.11% default rate across 500k loans nationally. The two together represent both failure modes of small samples.*

Query 3B (BRD): Default Behavior Analysis

---

- Identify customers with at least one defaulted loan for collections team outreach.
- **Expected Output Columns:** Customer ID, customer name, contact info (email/phone), total loans, defaulted loans
- **Business Question:** Who has defaulted and needs to be contacted?
- **Consideration:** Do you need DISTINCT customer_id or are duplicate rows acceptable?

In [7]:
query = """
WITH defaulted_customers AS (
    SELECT
        c.customer_id,
        c.full_name,
        c.phone_number,
        c.email_address,
        COUNT(l.loan_id) AS total_loans,
        SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS defaulted_count
    FROM customers c
    INNER JOIN loans l 
        ON c.customer_id = l.customer_id
    GROUP BY c.customer_id, c.full_name, c.phone_number, c.email_address
    HAVING SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) > 0
)
SELECT
    customer_id AS `Customer ID`,
    full_name AS `Customer Name`,
    total_loans AS `Loans`,
    defaulted_count AS `Defaulted Loans`,
    CONCAT(ROUND(defaulted_count * 100.0 / total_loans, 2), ' %') AS `Default Rate (%)`,
    phone_number AS `Contact`,
    email_address AS `Email`
FROM defaulted_customers
ORDER BY defaulted_count DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------+------------------+-----+---------------+----------------+------------+--------------------------------+
|Customer ID|Customer Name     |Loans|Defaulted Loans|Default Rate (%)|Contact     |Email                           |
+-----------+------------------+-----+---------------+----------------+------------+--------------------------------+
|1047       |Raghav Dubey      |3    |3              |100.00 %        |918494902096|raghav.dubey194@yahoo.com       |
|4620       |Pushti Doctor     |3    |3              |100.00 %        |918092197558|pushti.doctor209@outlook.com    |
|4724       |Ishani Joshi      |3    |3              |100.00 %        |917231771111|ishani.joshi420@outlook.com     |
|3821       |Dakshesh Purohit  |3    |2              |66.67 %         |919054366215|dakshesh.purohit663@hotmail.com |
|780        |Liam Kari         |3    |2              |66.67 %         |917791584189|liam.kari793@hotmail.com        |
|3728       |Jai Tailor        |3    |2              |66

STEP 3B (Workbook) — CIBIL Score Distribution Among Defaulters

---

What you're doing: Look at CIBIL score ranges for customers who have defaulted. Do most defaulters have low CIBIL scores — or are there high-CIBIL defaulters too? This distinction changes the collection approach entirely.

*Hint: Filter to defaulted customers. Show CIBIL score brackets. COUNT customers per bracket. What % high-CIBIL vs low-CIBIL?*

In [36]:
query = """
WITH cibil_data AS (   
    SELECT
        c.customer_id,
        c.cibil_score,
        c.full_name,
        l.loan_status,
        CASE 
            WHEN c.cibil_score >= 750 AND c.cibil_score <= 900 THEN 'Excellent (750-900)'
            WHEN c.cibil_score >= 650 THEN 'Good (650-749)'
            WHEN c.cibil_score >= 550 THEN 'Fair (550-649)'
            ELSE 'Poor (<550)'
        END AS `CIBIL Brackets`
    FROM customers c
    INNER JOIN loans l 
        ON c.customer_id = l.customer_id
)
SELECT 
    `CIBIL Brackets`,
    COUNT(*) AS `Total Loans`,
    SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS `Defaulted Loans`,
    CONCAT(ROUND(SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2), '%') AS `Default Rate (%)`
FROM cibil_data
GROUP BY `CIBIL Brackets`
ORDER BY SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-------------------+-----------+---------------+----------------+
|     CIBIL Brackets|Total Loans|Defaulted Loans|Default Rate (%)|
+-------------------+-----------+---------------+----------------+
|     Good (650-749)|       2364|            250|          10.58%|
|     Fair (550-649)|       1218|            217|          17.82%|
|Excellent (750-900)|       1278|             84|           6.57%|
|        Poor (<550)|        140|             39|          27.86%|
+-------------------+-----------+---------------+----------------+

Time: 0.211 seconds


> Define your CIBIL brackets now. You'll need them for Step 3E. Common brackets: 300–550 (Poor), 551–700 (Fair), 701–750 (Good), 751+ (Excellent).


Query 3C (BRD): High-Value Defaulter Identification

---

- Calculate total defaulted amount per customer and focus on high-value recovery targets.
- **Expected Output Columns:** Customer ID, customer name, contact info, total defaulted amount
- **Business Question:** Which customers owe us the most money?
- **Filter:** Focus on customers with total default exposure > ₹5 Lakhs (BRD requirement)
- **Consideration:** Should you use HAVING or WHERE for the ₹5L filter? (Think about aggregation timing)

In [9]:
query = """
SELECT 
    `Customer ID`,
    `Customer Name`,
    `Contact`,
    `Loan Amount`,
    `Defaulted Amount`
FROM (
    SELECT
        c.customer_id AS `Customer ID`,
        c.full_name AS `Customer Name`,
        c.phone_number AS `Contact`,
        CONCAT(ROUND(SUM(l.loan_amount) / 100000.0, 2), ' L') AS `Loan Amount`,
        CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) / 100000.0, 2), ' L') AS `Defaulted Amount`,
        SUM(l.loan_amount) AS loan_sum
    FROM customers c
    INNER JOIN loans l 
        ON c.customer_id = l.customer_id
    GROUP BY c.customer_id, c.full_name, c.phone_number
    HAVING SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) / 100000.0 > 5.0
)
ORDER BY loan_sum DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------+-------------------+------------+-----------+----------------+
|Customer ID|      Customer Name|     Contact|Loan Amount|Defaulted Amount|
+-----------+-------------------+------------+-----------+----------------+
|       4075|     Jalsa Zacharia|919851386292|    21.06 L|           8.4 L|
|        611|     Ranveer Badami|917655595635|    19.64 L|           8.5 L|
|       3728|         Jai Tailor|917563087040|    18.45 L|         12.91 L|
|        713|      Amara Sawhney|917222500976|    18.26 L|          9.91 L|
|       2856|       Nakul Narain|917584483762|    18.24 L|          6.84 L|
|       1702|Vaishnavi Chaudhari|919432421991|    17.72 L|          8.18 L|
|       3298|  Yashasvi Sundaram|917611212814|    17.46 L|         12.97 L|
|        648|    Tanvi Ahluwalia|918349781978|    17.42 L|          7.14 L|
|        165|    Ishaan Upadhyay|919761746556|    17.36 L|           8.6 L|
|       1651|       Brinda Mital|919851629927|    17.13 L|          7.85 L|
|       3473

STEP 3C (Workbook) — Open Metric Design: "Risk Efficiency"

---

What you're doing: The CEO doesn't want another "Top 10" list. They want a metric that reveals which segments are "burning cash" the fastest. You must define the components of this metric.

- Define your 'Risk Efficiency' Metric logic:

*Risk Efficiency = (Defaulted Loan Amount / Total Loan Amount) × log(N)*

Risk Efficiency is better than raw % because it adjusts for sample size using log(N), making results from larger, more reliable segments carry more weight. It also reflects business impact, not just risk rate, by prioritizing segments that are both risky and significant in scale. This makes it more useful for decision-making than plain default percentage.

- SQL Implementation:

In [28]:
query = """
WITH cibil_data AS (   
    SELECT
        l.loan_status,
        l.loan_amount,
        CASE 
            WHEN c.cibil_score >= 750 AND c.cibil_score <= 900 THEN 'Excellent (750-900)'
            WHEN c.cibil_score >= 650 THEN 'Good (650-749)'
            WHEN c.cibil_score >= 550 THEN 'Fair (550-649)'
            ELSE 'Poor (<550)'
        END AS `CIBIL Brackets`
    FROM customers c
    INNER JOIN loans l 
        ON c.customer_id = l.customer_id
)
SELECT 
    `CIBIL Brackets`,
    ROUND(SUM(CASE WHEN loan_status = 'Defaulted' THEN loan_amount * 1.0 ELSE 0 END) / 
    SUM(loan_amount) * LOG(COUNT(*)), 2) AS `Risk Efficiency`
FROM cibil_data
GROUP BY `CIBIL Brackets`
ORDER BY `Risk Efficiency` DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-------------------+---------------+
|     CIBIL Brackets|Risk Efficiency|
+-------------------+---------------+
|        Poor (<550)|           1.41|
|     Fair (550-649)|           1.27|
|     Good (650-749)|           0.82|
|Excellent (750-900)|            0.5|
+-------------------+---------------+

Time: 0.484 seconds


Query 3D (BRD): Demographic Risk Correlation

---

- Analyze if certain income brackets or employment types have higher default rates.
- **Expected Output Columns:** Employment type, customer count, default rate, portfolio size
- **Business Question:** Which employment types are highest risk?
- **Enhancement:** Adding income bracket dimension provides deeper insight (BRD mentions both)

In [11]:
query = """
WITH customer_base AS (
    SELECT
        customer_id,
        employemnet_type,
        CAST(annual_income AS DOUBLE) AS annual_income
    FROM customers
),

bucketed AS (
    SELECT
        *,
        NTILE(3) OVER (ORDER BY annual_income) AS income_bucket
    FROM customer_base
)

SELECT
    income_bucket,
    CONCAT(ROUND(MIN(annual_income) / 100000.0, 2), ' L') AS min_annual_income,
    CONCAT(ROUND(MAX(annual_income) / 100000.0, 2), ' L') AS max_annual_income,
    CONCAT(ROUND(AVG(annual_income) / 100000.0, 2), ' L') AS avg_annual_income,
    COUNT(*) AS customer_count
FROM bucketed
GROUP BY income_bucket
ORDER BY income_bucket;
"""
start = time.perf_counter()
spark.sql(query).show(n=N, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-------------+-----------------+-----------------+-----------------+--------------+
|income_bucket|min_annual_income|max_annual_income|avg_annual_income|customer_count|
+-------------+-----------------+-----------------+-----------------+--------------+
|1            |0.29 L           |5.48 L           |3.7 L            |1667          |
|2            |5.48 L           |8.93 L           |7.05 L           |1667          |
|3            |8.93 L           |38.16 L          |13.33 L          |1666          |
+-------------+-----------------+-----------------+-----------------+--------------+

Time: 0.469 seconds


- Low Income: `< 3L`
- Lower-Mid: `3L - 7L`
- Upper-Mid: `7L - 15L`
- High Income: `> 15L`

In [12]:
query = """
WITH loan_agg AS (
    SELECT
        customer_id,
        SUM(CAST(loan_amount AS DOUBLE)) AS total_loan_amount,
        COUNT(*) AS loan_count,
        SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS default_loans
    FROM loans
    GROUP BY customer_id
),
customer_base AS (
    SELECT
        customer_id,
        employemnet_type,
        CAST(annual_income AS DOUBLE) AS annual_income
    FROM customers
),
bucketed AS (
    SELECT
        *,
        CASE
            WHEN annual_income < 300000 THEN 'Low Income (<3L)'
            WHEN annual_income >= 300000 AND annual_income < 700000 THEN 'Lower-Mid (3L-7L)'
            WHEN annual_income >= 700000 AND annual_income < 1500000 THEN 'Upper-Mid (7L-15L)'
            ELSE 'High Income (>15L)'
        END AS income_bracket
    FROM customer_base
),
final AS (
    SELECT
        b.customer_id,
        b.employemnet_type,
        b.income_bracket,
        COALESCE(l.total_loan_amount, 0) AS total_loan_amount,
        COALESCE(l.default_loans, 0) AS default_loans
    FROM bucketed b
    LEFT JOIN loan_agg l
        ON b.customer_id = l.customer_id
)
SELECT
    employemnet_type AS `Employment Type`,
    income_bracket AS `Income Bracket`,
    COUNT(*) AS `Customer Count`,
    CONCAT(
        ROUND(SUM(total_loan_amount) / 10000000.0, 2), ' Cr'
    ) AS `Portfolio Value`,
    CONCAT(
        ROUND(
            SUM(default_loans) * 100.0 / COUNT(*),
        2),
        '%'
    ) AS `Default Rate`
FROM final
GROUP BY employemnet_type, income_bracket
ORDER BY employemnet_type, income_bracket;
"""
start = time.perf_counter()
spark.sql(query).show(n=N, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-------------------+------------------+--------------+---------------+------------+
|Employment Type    |Income Bracket    |Customer Count|Portfolio Value|Default Rate|
+-------------------+------------------+--------------+---------------+------------+
|Business Owner     |High Income (>15L)|185           |7.99 Cr        |7.57%       |
|Business Owner     |Low Income (<3L)  |2             |0.14 Cr        |100.00%     |
|Business Owner     |Lower-Mid (3L-7L) |84            |3.58 Cr        |8.33%       |
|Business Owner     |Upper-Mid (7L-15L)|259           |11.78 Cr       |8.88%       |
|Government Employee|High Income (>15L)|34            |1.48 Cr        |0.00%       |
|Government Employee|Low Income (<3L)  |32            |1.21 Cr        |18.75%      |
|Government Employee|Lower-Mid (3L-7L) |481           |20.11 Cr       |7.28%       |
|Government Employee|Upper-Mid (7L-15L)|432           |16.81 Cr       |4.86%       |
|Private Employee   |High Income (>15L)|99            |4.45 Cr   

STEP 3D (Workbook) — Income and Employment Risk Profile

---

What you're doing: Cross-reference default behaviour with annual income and employment type. Does self-employed vs. salaried status correlate with default rate? Does income bracket matter? This determines whether EduFin's underwriting criteria need adjustment.

- SQL Query (Employment Type)

In [32]:
query = """
-- Analyze loan portfolio and defaults by employment type
WITH employment_stats AS (
    SELECT
        c.employemnet_type,
        SUM(l.loan_amount) / 10000000.0 AS portfolio_amount_cr,
        SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) / 10000000.0 AS defaulted_amount_cr,
        COUNT(*) AS total_loans,
        SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS default_count,
        ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS default_rate
    FROM customers c
    INNER JOIN loans l 
        ON c.customer_id = l.customer_id
    GROUP BY c.employemnet_type
)
SELECT
    employemnet_type AS `Employment Type`,
    CONCAT(ROUND(portfolio_amount_cr, 2), ' Cr') AS `Portfolio Amount`,
    CONCAT(ROUND(defaulted_amount_cr, 2), ' Cr') AS `Defaulted Amount`,
    total_loans AS `Loans`,
    default_count AS `Default Count`,
    CONCAT(default_rate, '%') AS `Default Rate (%)`
FROM employment_stats
ORDER BY default_count DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-------------------+----------------+----------------+-----+-------------+----------------+
|Employment Type    |Portfolio Amount|Defaulted Amount|Loans|Default Count|Default Rate (%)|
+-------------------+----------------+----------------+-----+-------------+----------------+
|Private Employee   |94.3 Cr         |11.2 Cr         |2291 |275          |12.0%           |
|Self Employed      |39.56 Cr        |6.91 Cr         |967  |168          |17.37%          |
|Government Employee|39.61 Cr        |2.75 Cr         |958  |62           |6.47%           |
|Business Owner     |23.5 Cr         |2.08 Cr         |554  |46           |8.3%            |
|Student            |7.86 Cr         |1.3 Cr          |230  |39           |16.96%          |
+-------------------+----------------+----------------+-----+-------------+----------------+

Time: 0.235 seconds


- SQL Query (Income Brackets)

In [31]:
query = """
-- Analyze loan portfolio and defaults by income brackets
WITH income_brackets AS (
    SELECT
        customer_id,
        CASE
            WHEN CAST(annual_income AS DOUBLE) < 300000 THEN 'Low Income (<3L)'
            WHEN CAST(annual_income AS DOUBLE) >= 300000 AND CAST(annual_income AS DOUBLE) <= 800000 THEN 'Mid (3L-7L)'
            ELSE 'High Income (>8L)'
        END AS income_bracket   
    FROM customers
),
portfolio_stats AS (
    SELECT
        ib.income_bracket,
        SUM(l.loan_amount) / 10000000.0 AS portfolio_amount_cr,
        SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) / 10000000.0 AS defaulted_amount_cr,
        COUNT(*) AS total_loans,
        SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS default_count,
        ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS default_rate
    FROM income_brackets ib
    INNER JOIN loans l
        ON ib.customer_id = l.customer_id
    GROUP BY ib.income_bracket
)
SELECT
    income_bracket AS `Income Bracket`,
    CONCAT(ROUND(portfolio_amount_cr, 2), ' Cr') AS `Portfolio Amount`,
    CONCAT(ROUND(defaulted_amount_cr, 2), ' Cr') AS `Defaulted Amount`,
    total_loans AS `Loans`,
    default_count AS `Default Count`,
    CONCAT(default_rate, '%') AS `Default Rate (%)`
FROM portfolio_stats
ORDER BY default_count DESC;
"""

start = time.perf_counter()
spark.sql(query).show(n=N, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------------+----------------+----------------+-----+-------------+----------------+
|Income Bracket   |Portfolio Amount|Defaulted Amount|Loans|Default Count|Default Rate (%)|
+-----------------+----------------+----------------+-----+-------------+----------------+
|Mid (3L-7L)      |103.82 Cr       |13.7 Cr         |2525 |327          |12.95%          |
|High Income (>8L)|84.81 Cr        |7.8 Cr          |2024 |189          |9.34%           |
|Low Income (<3L) |16.19 Cr        |2.73 Cr         |451  |74           |16.41%          |
+-----------------+----------------+----------------+-----+-------------+----------------+

Time: 0.249 seconds


- SQL Query (Employemnt + Income)

In [15]:
query = """
WITH loan_agg AS (
    SELECT
        customer_id,
        SUM(CAST(loan_amount AS DOUBLE)) AS total_loan_amount,
        COUNT(*) AS loan_count,
        SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS default_loans
    FROM loans
    GROUP BY customer_id
),
customer_base AS (
    SELECT
        customer_id,
        employemnet_type,
        CAST(annual_income AS DOUBLE) AS annual_income
    FROM customers
),
bucketed AS (
    SELECT
        *,
        CASE
            WHEN annual_income < 300000 THEN 'Low Income (<3L)'
            WHEN annual_income >= 300000 AND annual_income < 800000 THEN 'Mid Income (3L-8L)'
            ELSE 'High Income (>8L)'
        END AS income_bracket
    FROM customer_base
),
final AS (
    SELECT
        b.customer_id,
        b.employemnet_type,
        b.income_bracket,
        COALESCE(l.total_loan_amount, 0) AS total_loan_amount,
        COALESCE(l.default_loans, 0) AS default_loans
    FROM bucketed b
    LEFT JOIN loan_agg l
        ON b.customer_id = l.customer_id
)
SELECT
    employemnet_type AS `Employment Type`,
    income_bracket AS `Income Bracket`,
    COUNT(*) AS `Customer Count`,
    CONCAT(ROUND(SUM(total_loan_amount) / 10000000.0, 2), ' Cr') AS `Portfolio Value`,
    CONCAT(
        ROUND(
            SUM(default_loans) * 100.0 / COUNT(*),
        2),
        '%'
    ) AS `Default Rate`
FROM final
GROUP BY employemnet_type, income_bracket
ORDER BY SUM(total_loan_amount) DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-------------------+------------------+--------------+---------------+------------+
|Employment Type    |Income Bracket    |Customer Count|Portfolio Value|Default Rate|
+-------------------+------------------+--------------+---------------+------------+
|Private Employee   |Mid Income (3L-8L)|1303          |53.39 Cr       |13.28%      |
|Private Employee   |High Income (>8L) |838           |36.61 Cr       |10.74%      |
|Government Employee|Mid Income (3L-8L)|591           |24.46 Cr       |7.78%       |
|Self Employed      |Mid Income (3L-8L)|479           |19.88 Cr       |20.04%      |
|Business Owner     |High Income (>8L) |420           |18.75 Cr       |8.57%       |
|Self Employed      |High Income (>8L) |413           |15.51 Cr       |12.83%      |
|Government Employee|High Income (>8L) |356           |13.94 Cr       |2.81%       |
|Student            |Low Income (<3L)  |197           |6.38 Cr        |17.77%      |
|Business Owner     |Mid Income (3L-8L)|108           |4.61 Cr   

> Which employment type has the highest default rate? Is that the same type that has the highest total defaulted amount? If not, why does that difference matter?

Query 3E (BRD): Prioritized Collections Target List

---

- Generate final ranked list of customers for immediate collections action.
- **Expected Output:** Customer-level records with priority scoring
- **Business Question:** Who should the collections team call first?
- **Required Elements:**
    - Customer ID, name, contact information
    - Total defaulted amount per customer
    - Priority score (you design the formula)
    - TOP 50 customers for focused targeting
- **Risk Calculation:** Combine default amount + CIBIL score + default count into priority score
- **REQUIRED Text Summary:** Profile of the "Ideal High-Risk Borrower" to watch out for

In [20]:
query = """
WITH customer_defaults AS (
    SELECT
        c.customer_id,
        c.full_name,
        c.phone_number,
        c.email_address,
        c.cibil_score,
        COUNT(l.loan_id) AS total_loans,
        CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) / 100000.0, 2), ' L') AS default_amount,
        CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2), '%') AS default_rate
    FROM customers c
    JOIN loans l
        ON c.customer_id = l.customer_id
    GROUP BY
        c.customer_id,
        c.full_name,
        c.phone_number,
        c.email_address,
        c.cibil_score
),
scored AS (
    SELECT
        *,
        (
            CAST(REPLACE(default_amount, ' L', '') AS DOUBLE) *
            CAST(REPLACE(default_rate, '%', '') AS DOUBLE)
        ) AS score
    FROM customer_defaults
) 
SELECT
    customer_id AS `Customer ID`,
    full_name AS `Name`,
    phone_number AS `Contact`,
    email_address AS `Email`,
    default_amount AS `Defaulted Amount`,
    default_rate AS `Default Rate (%)`,
    cibil_score AS `CIBIL Score`,
    ROUND(score, 2) AS `Priority Score`
FROM scored
ORDER BY score DESC
LIMIT 50;
"""
start = time.perf_counter()
spark.sql(query).show(n=20, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------+-----------------+------------+--------------------------------+----------------+----------------+-----------+--------------+
|Customer ID|Name             |Contact     |Email                           |Defaulted Amount|Default Rate (%)|CIBIL Score|Priority Score|
+-----------+-----------------+------------+--------------------------------+----------------+----------------+-----------+--------------+
|1047       |Raghav Dubey     |918494902096|raghav.dubey194@yahoo.com       |15.07 L         |100.0%          |486        |1507.0        |
|4347       |Max Pillay       |919575772783|max.pillay122@hotmail.com       |11.7 L          |100.0%          |730        |1170.0        |
|1666       |Saumya Sarraf    |918731863798|saumya.sarraf606@yahoo.com      |11.51 L         |100.0%          |711        |1151.0        |
|2935       |Mohammed Mammen  |918820058128|mohammed.mammen602@outlook.com  |11.4 L          |100.0%          |557        |1140.0        |
|3102       |Balveer Swamy 

The highest-risk profile is a borrower with recurring defaults (2-3), elevated credit exposure, and weak repayment (10L - 15L) discipline despite varying income levels, making them a priority for immediate collections action.

STEP 3E (Workbook) — Prioritized Collections Target List (Final Output)

---

What you're doing: Produce the final top-50 collections list. Filter to defaulted customers. Rank by a Priority Score that you design — combining CIBIL score, total defaulted amount, and default count. The output must be operationally usable: include segment label and recommended action per customer.

*Hint: Full collections list: customer_name, total_defaulted_cr, default_count, cibil_score, city, borrower_segment, collection_approach, priority_score. ORDER BY priority_score DESC. LIMIT 50.*

In [34]:
query = """
WITH defaulted_customer_data AS (
    SELECT
        c.customer_id,
        c.full_name,
        c.phone_number,
        c.email_address,
        c.cibil_score,
        c.annual_income,
        SUM(l.loan_amount) AS total_loan_amount,
        SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0
                 ELSE 0
            END) AS defaulted_amount,
        SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS default_count
    FROM customers c
    INNER JOIN loans l
        ON c.customer_id = l.customer_id
    GROUP BY
        c.customer_id,
        c.full_name,
        c.phone_number,
        c.email_address,
        c.cibil_score,
        c.annual_income
    HAVING SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) > 0
),
risk_score_data AS (
    SELECT
        *,
        ROUND(
            0.5 * (defaulted_amount / 100000.0)
            + 0.3 * default_count * 100000
            + 0.2 * (900 - cibil_score) * 1000,
        2) AS risk_score
    FROM defaulted_customer_data
),
risk_segments AS (
    SELECT
        *,
        AVG(risk_score) OVER () AS avg_risk,
        NTILE(3) OVER (ORDER BY risk_score DESC) AS risk_ntile,
        CASE
            WHEN NTILE(3) OVER (ORDER BY risk_score DESC) = 1 THEN 'High Risk'
            WHEN NTILE(3) OVER (ORDER BY risk_score DESC) = 2 THEN 'Medium Risk'
            ELSE 'Low Risk'
        END AS borrower_segment
    FROM risk_score_data
),
collection_strategy AS (
    SELECT
        *,
        CASE
            WHEN risk_score >= avg_risk THEN 'Above Average Risk'
            ELSE 'Below Average Risk'
        END AS relative_risk,
        CASE
            WHEN borrower_segment = 'High Risk' AND risk_score >= avg_risk THEN 'Immediate Legal Action'
            WHEN borrower_segment = 'High Risk' THEN 'Escalated Collection'
            WHEN borrower_segment = 'Medium Risk' AND risk_score >= avg_risk THEN 'Legal Notice'
            WHEN borrower_segment = 'Medium Risk' THEN 'Aggressive Collection'
            WHEN borrower_segment = 'Low Risk' AND risk_score >= avg_risk THEN 'Payment Plan Negotiation'
            ELSE 'Standard Recovery'
        END AS collection_approach
    FROM risk_segments
)
SELECT
    customer_id AS `Customer ID`,
    full_name AS `Customer Name`,
    phone_number AS `Contact`,
    email_address AS `Email`,
    CONCAT(ROUND(total_loan_amount / 100000.0, 2), ' L') AS `Loan Amount`,
    CONCAT(ROUND(defaulted_amount  / 100000.0, 2), ' L') AS `Default Amount`,
    cibil_score AS `CIBIL`,
    CONCAT(ROUND(annual_income / 100000.0, 2), ' L') AS `Annual Income`,
    default_count AS `Default Count`,
    risk_score AS `Risk Score`,
    borrower_segment AS `Borrower Segment`,
    relative_risk AS `Relative Risk`,
    collection_approach AS `Collection Approach`
FROM collection_strategy
ORDER BY risk_score DESC
LIMIT 50;
"""
start = time.perf_counter()
spark.sql(query).show(n=50, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------+------------------+------------+---------------------------------+-----------+--------------+-----+-------------+-------------+----------+----------------+------------------+----------------------+
|Customer ID|Customer Name     |Contact     |Email                            |Loan Amount|Default Amount|CIBIL|Annual Income|Default Count|Risk Score|Borrower Segment|Relative Risk     |Collection Approach   |
+-----------+------------------+------------+---------------------------------+-----------+--------------+-----+-------------+-------------+----------+----------------+------------------+----------------------+
|1047       |Raghav Dubey      |918494902096|raghav.dubey194@yahoo.com        |15.07 L    |15.07 L       |486  |2.26 L       |3            |172807.53 |High Risk       |Above Average Risk|Immediate Legal Action|
|4724       |Ishani Joshi      |917231771111|ishani.joshi420@outlook.com      |9.68 L     |9.68 L        |583  |11.33 L      |3            |153404.84 |High 

### BUILD YOUR CUSTOM COLUMNS — Required for Wednesday Submission

---

> NEW TECHNIQUE — CTEs and Subqueries: Phase 1 = CASE. Phase 2 = Window Functions. Today you unlock WITH ... AS (Common Table Expressions) — SQL's pipeline builder. A CTE lets you name an intermediate result and reference it later. This is how production analysts write multi-step logic in cloud environments like Databricks.

- Column 1 — borrower_segment: Build a CTE pipeline that first computes a risk_score per customer (weighted formula combining default_count, CIBIL, income), then segments based on the computed score. You must use at least one WITH ... AS block.

- Column 2 — collection_approach: Write a second CTE or subquery that compares each customer's risk_score against the portfolio-wide average. Tag as "Above Average Risk" or "Below Average Risk". Then map segment + relative risk to an action.

Your CTE pipeline (WITH risk_scored AS ..., segmented AS ...):

In [18]:
query = """
WITH defaulted_customer_data AS (
    SELECT 
        c.customer_id,
        c.full_name,
        c.cibil_score,
        c.annual_income,
        SUM(l.loan_amount) AS total_loan_amount,
        SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) AS defaulted_amount,
        SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS default_count
    FROM customers c
    INNER JOIN loans l
        ON c.customer_id = l.customer_id
    GROUP BY c.customer_id, c.full_name, c.cibil_score, c.annual_income
    HAVING SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) > 0
),
risk_score_data AS (
    SELECT 
        *,
        ROUND(
            0.5 * (defaulted_amount / 100000.0) +   
            0.3 * default_count * 100000 +          
            0.2 * (900 - cibil_score) * 1000    ,    
        2) AS risk_score
    FROM defaulted_customer_data
),
risk_segments AS (
    SELECT 
        *,
        AVG(risk_score) OVER () AS avg_risk,
        NTILE(3) OVER (ORDER BY risk_score DESC) AS risk_ntile,
        CASE 
            WHEN NTILE(3) OVER (ORDER BY risk_score DESC) = 1 THEN 'High Risk'
            WHEN NTILE(3) OVER (ORDER BY risk_score DESC) = 2 THEN 'Medium Risk'
            ELSE 'Low Risk'
        END AS borrower_segment
    FROM risk_score_data
),
collection_strategy AS (
    SELECT 
        *,
        CASE 
            WHEN risk_score >= avg_risk THEN 'Above Average Risk'
            ELSE 'Below Average Risk'
        END AS relative_risk,
        CASE 
            WHEN borrower_segment = 'High Risk' AND risk_score >= avg_risk 
                THEN 'Immediate Legal Action'
            WHEN borrower_segment = 'High Risk' 
                THEN 'Escalated Collection'
            WHEN borrower_segment = 'Medium Risk' AND risk_score >= avg_risk 
                THEN 'Legal Notice'
            WHEN borrower_segment = 'Medium Risk' 
                THEN 'Aggressive Collection'
            WHEN borrower_segment = 'Low Risk' AND risk_score >= avg_risk 
                THEN 'Payment Plan Negotiation'
            ELSE 'Standard Recovery'
        END AS collection_approach
    FROM risk_segments
)
SELECT 
    customer_id AS `Customer ID`,
    full_name AS `Customer Name`,
    CONCAT(ROUND(total_loan_amount / 100000.0, 2), ' L') AS `Loan Amount`,
    CONCAT(ROUND(defaulted_amount / 100000.0, 2), ' L') AS `Default Amount`,
    cibil_score AS `CIBIL`,
    CONCAT(ROUND(annual_income / 100000.0, 2), ' L') AS `Annual Income`,
    default_count AS `Default Count`,
    risk_score AS `Risk Score`,
    borrower_segment AS `Borrower Segment`,
    relative_risk AS `Relative Risk`,
    collection_approach AS `Collection Approach`
FROM collection_strategy
ORDER BY risk_score DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=20, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------+---------------+-----------+--------------+-----+-------------+-------------+----------+----------------+------------------+----------------------+
|Customer ID|Customer Name  |Loan Amount|Default Amount|CIBIL|Annual Income|Default Count|Risk Score|Borrower Segment|Relative Risk     |Collection Approach   |
+-----------+---------------+-----------+--------------+-----+-------------+-------------+----------+----------------+------------------+----------------------+
|1047       |Raghav Dubey   |15.07 L    |15.07 L       |486  |2.26 L       |3            |172807.53 |High Risk       |Above Average Risk|Immediate Legal Action|
|4724       |Ishani Joshi   |9.68 L     |9.68 L        |583  |11.33 L      |3            |153404.84 |High Risk       |Above Average Risk|Immediate Legal Action|
|4620       |Pushti Doctor  |10.28 L    |10.28 L       |637  |5.8 L        |3            |142605.14 |High Risk       |Above Average Risk|Immediate Legal Action|
|1988       |Kala Mishra    |6.0 L

Your collection_approach (must reference portfolio average via subquery)

Why is a CTE pipeline better than a single nested CASE? (2-3 sentences):

CTEs are better because they break complex logic into readable, named stages rather than burying it in nested CASE statements. This makes debugging easier (you can inspect each CTE layer) and lets the database reuse intermediate results instead of recalculating them. 

### BEFORE YOU SUBMIT — Answer These Three Questions

---

1. If you added employment_type as a third factor in borrower_segment — would any customers move to a different segment? Pick one specific customer and show the re-classification logic.

In my current logic to calculating borrower segment, i have used the risk score (default_amount, default_count & annual income) and not CIBIL brackets. When employment_type is added, it creates a composite_risk_score through a weighted formula, which changes the customer ranking order, customers with better employment stability get lower composite scores and may shift from High Risk to Medium Risk if their new rank crosses the NTILE boundary.

2. Your top 50 list: the collections team can make 200 calls per week. How do you prioritise the order within your top 50? Is priority_score alone sufficient?

I have ordered my top 50 list using a priority_score calculated by multiplying defaulted_amount and default_rate for each customer. This is sufficient because the collections team makes 200 calls/week—exactly the capacity for a top 50 list—and the priority_score directly optimizes for recovery: customers with large outstanding defaults AND high default rates rank first, combining financial exposure with behavioral risk. The team works top-to-bottom knowing each customer represents maximum recovery potential, eliminating the need for additional manual prioritization or tie-breakers.

3. Write the "Riskiest Borrower Profile" paragraph. Describe the type of customer who appears most frequently in your top segment.

The riskiest borrower profile is typically a self-employed or student customer with a low to mid CIBIL score and multiple past defaults. These borrowers often have higher defaulted loan amounts and repeated default behavior, indicating weak repayment stability. They appear most frequently in the top risk segment due to both financial exposure and unstable income sources.

### Phase 3: Findings & Insights

---